# Token Feature Pair Scatterplots

Sample 100 HG and 100 MG texts for each dataset group, then compare all pairwise combinations of the four existing token-level features. No model inference is performed.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

from IPython.display import display

from notebooks.utils.token_feature_scatter_utils import (
    TOKEN_FEATURE_COLUMNS,
    load_balanced_token_feature_group_frame,
    plot_all_feature_pairs,
)

DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
PROCESSED_ROOT = DATA_ROOT / "processed"

## Parameters

Each group is sampled to 100 HG and 100 MG texts. `Mix` combines Kaggle, MAGE, and Outfox; `Mix+BBC` also includes BBC human texts.

In [ ]:
MODEL = "gemma_3_270m_it"
TEXTS_PER_SPLIT = 100
RANDOM_STATE = 42
PARQUET_BATCH_SIZE = 256
DATASET_GROUPS = {
    "Kaggle": ("kaggle_train",),
    "MAGE": ("mage_train",),
    "Outfox": ("outfox_chatgpt_train",),
    "Mix": ("kaggle_train", "mage_train", "outfox_chatgpt_train"),
    "Mix+BBC": ("kaggle_train", "mage_train", "outfox_chatgpt_train", "bbc_news_content"),
}
DATASET_SPLITS = {"bbc_news_content": ("HG",)}
print(f"Features: {', '.join(TOKEN_FEATURE_COLUMNS)}")

## Dataset Group Plots

Each point is one token. Colors identify HG and MG texts. Every group produces six pairwise scatterplots.

In [ ]:
for group_name, datasets in DATASET_GROUPS.items():
    token_features = load_balanced_token_feature_group_frame(
        PROCESSED_ROOT,
        datasets,
        MODEL,
        texts_per_split=TEXTS_PER_SPLIT,
        random_state=RANDOM_STATE,
        batch_size=PARQUET_BATCH_SIZE,
        dataset_splits=DATASET_SPLITS,
    )
    print(f"{group_name}:")
    display(token_features[["source_dataset", "text_id", "split"]].drop_duplicates().groupby("split").size().rename("texts"))
    figure, axes = plot_all_feature_pairs(token_features)
    figure.suptitle(f"{group_name} / {MODEL}", y=1.02)
    display(figure)